# Fake News Detection System
### End-to-End ML Pipeline with Dataset Generation, EDA, and Model Training

This notebook:
1. Generates a synthetic fake vs real news dataset
2. Performs Exploratory Data Analysis (EDA)
3. Preprocesses text data
4. Trains multiple classifiers
5. Evaluates and compares model performance

## Step 1: Install and Import Libraries

In [ ]:
# Install required libraries
!pip install -q wordcloud nltk scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import string
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

from wordcloud import WordCloud

print('All libraries imported successfully!')

## Step 2: Generate Synthetic Dataset
We programmatically create a labelled dataset of real and fake news headlines + body text.

In [ ]:
random.seed(42)
np.random.seed(42)

# --- REAL NEWS templates ---
real_headlines = [
    "Government announces new policy on renewable energy funding",
    "Scientists discover new species of deep-sea fish in Pacific Ocean",
    "Stock markets rise as inflation data shows signs of easing",
    "Health officials urge vaccination ahead of flu season",
    "Tech giant reports record quarterly earnings amid strong demand",
    "UN climate summit concludes with landmark emissions agreement",
    "Researchers develop new treatment for antibiotic-resistant bacteria",
    "Central bank raises interest rates to combat rising prices",
    "Local elections see record voter turnout across major cities",
    "Space agency successfully launches telescope into orbit",
    "Economists warn of slowdown as trade deficits widen",
    "Education ministry rolls out digital learning program nationwide",
    "Hospital reports decline in COVID-19 admissions this quarter",
    "Study finds regular exercise reduces risk of heart disease",
    "Aviation authority clears new aircraft model for commercial use",
    "Parliament passes budget with focus on infrastructure spending",
    "Wildlife conservation effort saves endangered tiger population",
    "Cybersecurity firm warns of rise in phishing attacks",
    "New trade agreement signed between two major economies",
    "Agricultural output rises due to improved irrigation technology"
]

real_bodies = [
    "According to official sources, the new policy will allocate significant funds toward solar and wind energy projects over the next five years.",
    "Marine biologists from the National Oceanic Institute published findings in the journal Nature this week, describing the newly documented species.",
    "The latest Consumer Price Index data released on Thursday showed a month-on-month decline in core inflation, boosting investor confidence.",
    "The Centers for Disease Control and Prevention released updated guidelines recommending annual flu shots for all adults over the age of six months.",
    "The company reported earnings per share of $4.82, beating analyst expectations by 12 percent, driven largely by cloud computing services.",
    "Delegates from over 190 countries agreed to a framework reducing carbon emissions by 45 percent relative to 2005 levels by 2035.",
    "The drug, developed using CRISPR gene-editing technology, successfully eliminated resistant strains of bacteria in clinical trials.",
    "The Federal Reserve raised the benchmark rate by 25 basis points, citing persistent price pressures in the services sector.",
    "Election commission data showed turnout exceeded 68 percent in metropolitan areas, the highest since records began in 1980.",
    "The orbital telescope, equipped with infrared imaging sensors, is designed to study exoplanets up to 100 light-years from Earth.",
    "A report from the IMF projected GDP growth of 2.1 percent for the current fiscal year, down from the earlier forecast of 3.4 percent.",
    "The program will provide tablets and high-speed internet access to approximately 2 million students in rural districts.",
    "Hospital administrators attributed the decline to widespread community vaccination and updated treatment protocols.",
    "The longitudinal study, which tracked 40,000 adults over 20 years, found a 32 percent reduction in cardiac events among regular exercisers.",
    "Following a rigorous 18-month review process, the regulator confirmed the aircraft met all safety standards required for passenger flights.",
    "The approved budget totals over $800 billion, with a third dedicated to road, bridge, and public transit improvements.",
    "A joint effort between three governments and a leading conservation NGO helped increase the Bengal tiger count by 20 percent.",
    "The firm identified a 300 percent increase in spear-phishing emails targeting financial institutions over the past quarter.",
    "The agreement eliminates tariffs on over 5,000 goods and is expected to boost bilateral trade by $200 billion annually.",
    "The Ministry of Agriculture credited drip irrigation initiatives funded under the national food security plan for the improved yields."
]

# --- FAKE NEWS templates ---
fake_headlines = [
    "SHOCKING: Government secretly adds mind-control chemicals to tap water",
    "Doctors DON'T want you to know this one cure for all diseases",
    "Billionaires planning to fake moon landing again to distract public",
    "5G towers proven to cause cancer, government covers up study",
    "Leaked documents reveal vaccines contain microchips from tech company",
    "BREAKING: Aliens landed in Nevada and government hiding the proof",
    "Scientists admit climate change is a hoax invented for profit",
    "Eating raw garlic cures diabetes permanently, study claims",
    "Deep state planning to ban all cash by end of year, insiders say",
    "Famous actor admits he was replaced by a robot clone in 2019",
    "The moon is actually hollow and used as a base by secret society",
    "GMO foods causing mass sterility, suppressed study reveals",
    "Chemtrails from planes contain chemicals to make people docile",
    "Natural herb found in Amazon cures all forms of cancer overnight",
    "Global elite secretly control weather using HAARP machines",
    "New law will force all citizens to be implanted with tracking device",
    "NASA images of Mars are all faked in a Hollywood studio",
    "Drinking bleach in small amounts boosts immunity, goes viral",
    "Mainstream media admits faking major news event for ratings",
    "Secret government project turns birds into surveillance drones"
]

fake_bodies = [
    "An anonymous whistleblower has leaked documents purportedly showing officials approved fluoride mixtures that alter brain function.",
    "A viral post claims a retired physician discovered a fruit-based remedy that pharmaceutical companies have been suppressing for decades.",
    "A forum post citing unnamed aerospace insiders alleges that the 2024 lunar mission footage was filmed in a studio in Arizona.",
    "A widely shared video claims a German scientist proved a link between 5G radiation and tumour growth, though no peer-reviewed study has been located.",
    "Social media posts circulating documents alleged to show nanotechnology devices embedded in mRNA vaccine vials.",
    "An unverified blog post claims satellite imagery shows metallic structures near Area 51 consistent with spacecraft hangars.",
    "A self-described professor posted a thread alleging that temperature data from all major research institutions has been systematically altered.",
    "A YouTube video promoted a study supposedly from a university in Asia claiming daily garlic consumption reversed Type 2 diabetes in 30 days.",
    "A Telegram channel with millions of followers posted alleged central bank memos discussing eliminating physical currency to track spending.",
    "A tabloid claims insiders from a major film studio confirmed the real actor disappeared and was substituted following a secret meeting.",
    "An online documentary claims sonar imaging reveals artificial chambers inside the lunar surface used for undisclosed purposes.",
    "An anonymously published PDF claims a suppressed EPA study found significant reproductive harm in populations consuming genetically engineered corn.",
    "A widely shared video alleges flight logs from commercial aircraft confirm regular dispersal of undisclosed compounds.",
    "A viral health blog claims an indigenous tribe uses a specific plant extract that clinical tests allegedly showed destroys cancer cells in 48 hours.",
    "An anonymous intelligence leaker claims military facilities in Alaska are used to trigger hurricanes and droughts on demand.",
    "A chain email claims a bill currently in a secret legislative session will mandate subcutaneous RFID chip implantation for all adults.",
    "A conspiracy channel claims former NASA employees confirmed that all Mars surface imagery is generated using CGI software.",
    "A now-deleted post advised followers to ingest diluted bleach, claiming it activates white blood cell production.",
    "A leaked private video allegedly shows a prominent anchor admitting to fabricating a crisis story to improve network ratings.",
    "A viral claim states that birds went extinct in 1986 and the government replaced them with robotic models for population surveillance."
]

def expand_dataset(headlines, bodies, label, n=500):
    rows = []
    base = list(zip(headlines, bodies))
    intensifiers = ["", "EXCLUSIVE: ", "URGENT: ", "REVEALED: ", "UPDATE: "]
    suffixes = ["", " - full report", " - experts react", " - what they hide", " - the truth"]
    extra_phrases = [
        "Reports indicate that ", "Sources confirm ", "Analysts believe ",
        "New evidence shows ", "Officials state that ", "Data suggests ",
        "Experts argue ", "Investigations reveal "
    ]
    for i in range(n):
        h, b = random.choice(base)
        prefix = random.choice(intensifiers) if label == 0 else ""
        suffix = random.choice(suffixes) if label == 0 else ""
        headline = prefix + h + suffix
        extra = random.choice(extra_phrases)
        body = extra + b + " " + random.choice(["Further details are awaited.", "No official response yet.", "This story is developing."])
        rows.append({"title": headline, "text": body, "label": label})
    return rows

real_rows = expand_dataset(real_headlines, real_bodies, label=1, n=600)
fake_rows = expand_dataset(fake_headlines, fake_bodies, label=0, n=600)

df = pd.DataFrame(real_rows + fake_rows).sample(frac=1, random_state=42).reset_index(drop=True)
df['label_name'] = df['label'].map({1: 'REAL', 0: 'FAKE'})

print(f'Dataset shape: {df.shape}')
print(df['label_name'].value_counts())
df.head()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#e74c3c', '#2ecc71']
counts = df['label_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Text length distribution
df['text_length'] = df['text'].apply(len)
df['title_length'] = df['title'].apply(len)

for label, color in zip(['REAL', 'FAKE'], colors):
    subset = df[df['label_name'] == label]['text_length']
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='black', linewidth=0.3)

axes[1].set_title('Text Length Distribution by Label', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df.groupby('label_name')['text_length'].describe().round(1))

In [ ]:
# Word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, color_map in zip(axes, ['REAL', 'FAKE'], ['Greens', 'Reds']):
    text_data = ' '.join(df[df['label_name'] == label]['title'] + ' ' + df[df['label_name'] == label]['text'])
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap=color_map, max_words=100,
                   stopwords=set(stopwords.words('english'))).generate(text_data)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{label} News - Word Cloud', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 4: Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)         # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()     # normalize whitespace
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['combined'] = df['title'] + ' ' + df['text']
df['processed'] = df['combined'].apply(preprocess_text)

print('Sample processed text:')
print(df[['label_name', 'processed']].head(3).to_string())

## Step 5: Feature Engineering & Train/Test Split

In [ ]:
X = df['processed']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)} samples')
print(f'Test set:     {len(X_test)} samples')
print(f'Train label distribution:\n{y_train.value_counts().rename({1: "REAL", 0: "FAKE"})}')

## Step 6: Train Multiple Classifiers

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Passive Aggressive': PassiveAggressiveClassifier(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=2)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    trained_models[name] = (model, preds)
    print(f'{name:25s} -> Accuracy: {acc:.4f}')

## Step 7: Model Comparison & Evaluation

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
accs = list(results.values())
bars = ax.barh(names, accs, color=plt.cm.viridis(np.linspace(0.2, 0.8, len(names))), edgecolor='black')
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.002, bar.get_y() + bar.get_height()/2,
            f'{acc:.4f}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Best model - detailed report
best_name = max(results, key=results.get)
best_model, best_preds = trained_models[best_name]

print(f'Best Model: {best_name} (Accuracy: {results[best_name]:.4f})')
print('=' * 55)
print(classification_report(y_test, best_preds, target_names=['FAKE', 'REAL']))

In [ ]:
# Confusion matrix for best model
cm = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['FAKE', 'REAL'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8: Top Predictive Features

In [ ]:
# Use Logistic Regression for interpretability
lr_model = trained_models['Logistic Regression'][0]
feature_names = tfidf.get_feature_names_out()
coefs = lr_model.coef_[0]

top_n = 15
top_fake_idx = np.argsort(coefs)[:top_n]
top_real_idx = np.argsort(coefs)[-top_n:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(feature_names[top_fake_idx], coefs[top_fake_idx], color='#e74c3c', edgecolor='black')
axes[0].set_title('Top Features -> FAKE News', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Coefficient Value')

axes[1].barh(feature_names[top_real_idx], coefs[top_real_idx], color='#2ecc71', edgecolor='black')
axes[1].set_title('Top Features -> REAL News', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

## Step 9: Predict on Custom Text

In [ ]:
def predict_news(text, model=best_model, vectorizer=tfidf):
    processed = preprocess_text(text)
    vec = vectorizer.transform([processed])
    pred = model.predict(vec)[0]
    label = 'REAL' if pred == 1 else 'FAKE'
    print(f'Input : {text[:100]}')
    print(f'Result: [{label}] News')
    print('-' * 60)

# Test with sample headlines
test_texts = [
    "Scientists announce major breakthrough in Alzheimer's treatment after 10-year study",
    "SHOCKING: Government secretly poisoning water supply with mind control drugs",
    "Central bank holds interest rates steady amid mixed economic data",
    "Doctors HATE this one trick that cures cancer in three days"
]

print(f'Using model: {best_name}\n')
for t in test_texts:
    predict_news(t)

## Step 10: Save the Best Model

In [ ]:
import pickle

with open('fake_news_model.pkl', 'wb') as f:
    pickle.dump({'model': best_model, 'vectorizer': tfidf, 'model_name': best_name}, f)

print(f'Model saved: fake_news_model.pkl')
print(f'Best Model : {best_name}')
print(f'Accuracy   : {results[best_name]:.4f}')